# 02 - Model Training with XGBoost

This notebook trains an XGBoost classifier for customer churn prediction and registers it in the Snowflake Model Registry with tracked metrics.

**Key Concepts**:
- Training on Container Runtime (no infrastructure management)
- Metric logging with `model_version.set_metric()`
- Automatic lineage capture via `sample_input_data`
- Model versioning in the registry

**Prerequisites**: Run `00_data_setup.ipynb` first.

In [ ]:
from snowflake.snowpark.context import get_active_session
session = get_active_session()
print(f"Connected: {session.get_current_user()} | {session.get_current_role()}")

In [ ]:
import xgboost as xgb
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
)
from snowflake.ml.registry import Registry

print(f"XGBoost version: {xgb.__version__}")

In [ ]:
# Load data
df = session.table("MLOPS_DEMO_DB.RAW.CUSTOMERS").to_pandas()

feature_cols = [
    "CREDIT_SCORE", "AGE", "TENURE", "BALANCE",
    "NUM_OF_PRODUCTS", "HAS_CR_CARD", "IS_ACTIVE_MEMBER", "ESTIMATED_SALARY"
]
target_col = "EXITED"

X = df[feature_cols]
y = df[target_col]

print(f"Features: {X.shape}")
print(f"Target distribution:\n{y.value_counts(normalize=True).round(3)}")

In [ ]:
# Train/test split with stratification to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} samples")
print(f"Test:  {X_test.shape[0]} samples")

In [ ]:
# Train XGBoost classifier
model = xgb.XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

model.fit(X_train, y_train)
print("Model training complete.")

In [ ]:
# Evaluate the model
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": round(accuracy_score(y_test, y_pred), 4),
    "f1": round(f1_score(y_test, y_pred), 4),
    "precision": round(precision_score(y_test, y_pred), 4),
    "recall": round(recall_score(y_test, y_pred), 4),
    "auc_roc": round(roc_auc_score(y_test, y_proba), 4),
    "train_samples": X_train.shape[0],
    "test_samples": X_test.shape[0],
    "n_features": X_train.shape[1]
}

print("Model Metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v}")

In [ ]:
# Register model in Snowflake Model Registry
reg = Registry(
    session=session,
    database_name="MLOPS_DEMO_DB",
    schema_name="PIPELINES"
)

# sample_input_data enables automatic lineage capture
sample_input = X_train.head(10)

mv = reg.log_model(
    model,
    model_name="CHURN_PREDICTION_MODEL",
    version_name="v1",
    sample_input_data=sample_input,
    comment="XGBoost churn prediction model - initial version"
)
print(f"Model registered: {mv.model_name} version {mv.version_name}")

In [ ]:
# Log all metrics to the model version
for metric_name, metric_value in metrics.items():
    mv.set_metric(metric_name, metric_value)
    print(f"  Logged: {metric_name} = {metric_value}")

print("\nAll metrics logged to Model Registry.")

In [ ]:
# Verify metrics are stored
print("Stored metrics:")
mv.show_metrics()

In [ ]:
# Feature importance from XGBoost
importance = model.feature_importances_
feat_imp = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": importance
}).sort_values("Importance", ascending=False)

print("Feature Importance (XGBoost gain):")
print(feat_imp.to_string(index=False))

## What to Show in SnowSight

Navigate to **AI/ML > Model Registry** to see:
- `CHURN_PREDICTION_MODEL` with version `v1`
- Click on the version to see logged metrics (accuracy, f1, precision, recall, auc_roc)
- The **Files** tab shows the serialized model artifacts
- The **Lineage** tab shows the connection from source data to this model

---

**Next**: `03_hyperparameter_tuning.ipynb` - Distributed hyperparameter optimization